# SAM + RGB-D Pipeline Debug

Notebook para testar a pipeline por etapas usando os modulos de `src/franka_perception`.

In [7]:
import matplotlib.pyplot as plt

from franka_perception.pipeline import CubeDetectionPipeline
from franka_perception.rgbd_inputs import load_depth_image, load_rgb_image
from franka_perception.rgbd_masking import CameraIntrinsics, erode_binary_masks, masked_depth_to_point_cloud_clusters
from franka_perception.sam_rgbd_pipeline import SamRgbdCubePipeline
from franka_perception.sam_segmentation import SamSegmenter
from franka_perception.sam_visualization import overlay_masks
from franka_perception.visualization import draw

ModuleNotFoundError: No module named 'franka_perception'

In [ ]:
# Ajuste estes caminhos/parametros
RGB_PATH = '/tmp/rgb.png'
DEPTH_PATH = '/tmp/depth.png'
SAM_CHECKPOINT = '/tmp/sam_vit_b.pth'
SAM_MODEL_TYPE = 'vit_b'
SAM_DEVICE = 'cuda'
DEPTH_SCALE = 1000.0  # 1000 para depth em mm, 1.0 para metros
DEPTH_TRUNC = 3.0
INTRINSICS = CameraIntrinsics(
    fx=527.2972393595844,
    fy=527.2972393595844,
    cx=658.8206787109375,
    cy=372.25762939453125,
    width=1280,
    height=720,
)

In [ ]:
rgb = load_rgb_image(RGB_PATH)
depth = load_depth_image(DEPTH_PATH)
print('RGB:', rgb.shape, rgb.dtype)
print('Depth:', depth.shape, depth.dtype)

In [ ]:
segmenter = SamSegmenter(
    checkpoint_path=SAM_CHECKPOINT,
    model_type=SAM_MODEL_TYPE,
    device=SAM_DEVICE,
    points_per_side=32,
    pred_iou_thresh=0.86,
    stability_score_thresh=0.92,
    min_mask_region_area=100,
)
sam_masks = segmenter.generate_masks(rgb, min_area_pixels=200, max_masks=20)
print('SAM masks:', len(sam_masks))

In [ ]:
raw_overlay = overlay_masks(rgb, [m.mask for m in sam_masks])
eroded_masks = erode_binary_masks([m.mask for m in sam_masks], kernel_size=3, iterations=1)
eroded_overlay = overlay_masks(rgb, eroded_masks)

plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
plt.title('SAM masks (raw)')
plt.imshow(raw_overlay)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title('SAM masks (eroded)')
plt.imshow(eroded_overlay)
plt.axis('off')
plt.show()

In [ ]:
clusters, cluster_points = masked_depth_to_point_cloud_clusters(
    depth,
    eroded_masks,
    INTRINSICS,
    depth_scale=DEPTH_SCALE,
    depth_trunc=DEPTH_TRUNC,
    min_points=30,
    flip=True,
)
print('3D clusters:', len(clusters))
print('Points per cluster:', [p.shape[0] for p in cluster_points])

In [ ]:
cube_pipeline = CubeDetectionPipeline(
    cube_side_length=0.045,
    voxel_size=0.002,
    base_plane_distance=0.01,
    cluster_eps=0.005,
    cluster_min_points=10,
    max_cubes_per_cluster=2,
    clearance=0.015,
)
result_from_clusters = cube_pipeline.process_preclustered_clusters(clusters, stop_after='all')
print('Cubos detectados:', len(result_from_clusters.cubes))

In [ ]:
draw(result_from_clusters, axis_size=0.1)

In [ ]:
# Execucao completa via orquestrador unico
sam_rgbd_pipeline = SamRgbdCubePipeline(cube_pipeline)
result_full, debug_full = sam_rgbd_pipeline.process(
    rgb_image=rgb,
    depth_image=depth,
    intrinsics=INTRINSICS,
    segmenter=segmenter,
    stop_after='all',
    min_area_pixels=200,
    max_masks=20,
    erosion_kernel=3,
    erosion_iterations=1,
    depth_scale=DEPTH_SCALE,
    depth_trunc=DEPTH_TRUNC,
    min_cluster_points=30,
    flip=True,
)
print('Cubos detectados (pipeline completa):', len(result_full.cubes))